In [1]:
import pandas as pd
import numpy as np

In [23]:
DIRECTORY = "/Users/jowanglin/Downloads"
template = pd.read_excel(f"{DIRECTORY}/aggregatePitch.xlsx")
STUDENTS = template["Student"].to_list()
N_COLS = template.shape[1] - 1

In [41]:
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    def per_cell(x):
        if x in STUDENTS:
            return x
        elif isinstance(x, int) or isinstance(x, float):
            if 1.0 <= x <= 4.0:
                return x
            else:
                return np.nan
        else:
            return np.nan
    df = df.map(per_cell)
    return df


def get_avg(file_name: str) -> pd.DataFrame:
    file = pd.ExcelFile(f"{DIRECTORY}/{file_name}.xlsx")
    graders = file.sheet_names

    avg_data = np.full((len(graders), len(STUDENTS), N_COLS), np.nan)
    for i, grader in enumerate(graders):
        df = pd.read_excel(f"{DIRECTORY}/{file_name}.xlsx", sheet_name=grader)
        df = clean_data(df)
        data = df.to_numpy()[:, 1:]
        avg_data[i, :, :] = data
    avg_data = np.nanmean(avg_data, axis=0)

    table = pd.DataFrame(data=avg_data, columns=list(df.columns)[1:])
    table.insert(0, "Student", STUDENTS)
    table = table.reset_index(drop=True)
    return table

table_pitch = get_avg("aggregatePitch")
table_final = get_avg("aggregateFinal")

